# 01 — Origem: PostgreSQL

Le uma tabela de um Postgres **externo** e grava na camada `coleta` como tabela Iceberg.

A camada `coleta` guarda o dado **como veio**: sem renomear coluna, sem converter tipo, sem filtrar.
A limpeza acontece no notebook `10_tratamento`. Manter a coleta fiel a origem e o que permite
reprocessar sem voltar ao banco.

In [ ]:
from lakehouse import sessao, ler_jdbc, gravar, perfil

spark = sessao("01-postgres")

## Conexao

Preencha com os dados do **seu** banco. O Postgres que acompanha a stack esta abaixo como exemplo —
de dentro da rede Docker ele atende em `postgres:5432`.

> Nao deixe senha no notebook se ele for versionado. Prefira variavel de ambiente:
> `senha = os.environ["SENHA_ORIGEM"]`.

In [ ]:
ORIGEM = dict(
    tipo="postgres",
    host="postgres",          # host ou IP do banco
    porta=5432,
    banco="ssp_db",
    usuario="postgres_admin",
    senha="postgres-local-2026",
)

TABELA_ORIGEM = "public.customers"     # ou uma subconsulta, ver abaixo
DESTINO = "coleta.clientes"

In [ ]:
df = ler_jdbc(spark, tabela=TABELA_ORIGEM, **ORIGEM)
perfil(df)

### Trazer so parte da tabela

Em vez do nome da tabela, passe uma subconsulta entre parenteses com alias. O filtro roda **no
banco de origem**, entao o Spark nao puxa o que nao precisa:

```python
df = ler_jdbc(spark, tabela="(SELECT * FROM vendas WHERE data >= '2026-01-01') AS v", **ORIGEM)
```

### Tabela grande: leitura em paralelo

Sem isso, o Spark le tudo por **uma unica conexao**. Escolha uma coluna numerica bem distribuida:

```python
df = ler_jdbc(spark, tabela="public.vendas", **ORIGEM,
              partitionColumn="id", lowerBound=1, upperBound=5_000_000, numPartitions=8)
```

`fetchsize=10000` tambem ajuda em tabelas largas.

## Gravar na camada de coleta

`substituir` recria a tabela a cada execucao — e o certo para carga cheia.
Para carga incremental, use `acrescentar` junto com um filtro por data na origem.

In [ ]:
gravar(df, DESTINO, modo="substituir")

In [ ]:
spark.sql(f"SELECT * FROM nessie.{DESTINO} LIMIT 5").show(truncate=40)

Pronto. A tabela ja aparece no Dremio em `nessie → coleta → clientes`.

Proximo passo: **`10_tratamento.ipynb`**.